# GPT-2 3D Semantic & Mechanistic Concept Map

Trace neuron clusters across token positions and layer depths during a single GPT-2 forward pass.

## What this notebook does

1. **Multi-layer + multi-token extraction**: For one input sentence, capture activations at every (token position, transformer block) pair from GPT-2 small (12 layers × 768 dims).
2. **Feature decoding**: Run each activation vector through `HypoSpaceAPI.decode()` to extract top-k semantic features.
3. **3D concept map**: Visualize all features in (token × layer × score) space, colored by intensity band, sized by mechanistic effect size.
4. **Neuron cluster identification**: Group features by `source_index` (the neuron dimension) and rank clusters by total activation across all positions.
5. **Temporal evolution**: Track each top cluster's score trajectory across token positions and across layer depth.
6. **Interactive cluster isolation**: Drop down to inspect any cluster's per-(layer, token) heatmap.

## 4D encoding for the 3D scatter

| Axis | Encoding |
|------|----------|
| X | Token position (time during inference) |
| Y | Layer index (depth) |
| Z | Feature score (max-abs normalized) |
| Color | Intensity band (high / medium / low) |
| Size | Mechanistic effect size |

If `nnsight` / `torch` aren't installed, all GPT-2 cells skip gracefully.

In [ ]:
# Run once per environment (skip if already installed)
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q nnsight diskcache plotly pandas ipywidgets transformers

In [ ]:
import sys
import os
import warnings
import statistics

# Make the HypoSpace package importable from the notebooks/ subdir.
for _p in [os.path.abspath(".."), os.path.abspath(".")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from api import HypoSpaceAPI
from core.config import DecoderConfig, RuntimeConfig, GovernanceConfig
from viz.canvas import SemanticCanvas
from diagnostics import run_diagnostics
from interpretability.mechanistic import MechanisticAnalyzer

try:
    from data.nnsight_extractor import NNSightExtractor
    from transformers import AutoTokenizer
    NNSIGHT_AVAILABLE = True
    print("nnsight + transformers available")
except ImportError:
    NNSIGHT_AVAILABLE = False
    print("nnsight not available - GPT-2 cells will be skipped")
    print("Install with: pip install torch nnsight transformers")

print("Imports complete")

## Configuration

All run parameters live in this single cell. Adjust `INPUT_TEXT` to explore different sentences,
`TOP_K` to control how many features are decoded per (token, layer), or `TOP_N_CLUSTERS` to
control how many clusters are surfaced in the trajectory plots.

In [ ]:
INPUT_TEXT     = "The quick brown fox jumps"
MODEL_NAME     = "gpt2"
N_LAYERS       = 12
TOP_K          = 8
TOP_N_CLUSTERS = 10
CACHE_DIR      = "../.hypo_cache"
VERSION        = "0.1.0"

LAYER_PATHS  = [f"transformer.h.{i}" for i in range(N_LAYERS)]
LAYER_LABELS = [f"h.{i}" for i in range(N_LAYERS)]
LAYER_IDS    = [f"gpt2-h{i}" for i in range(N_LAYERS)]

INTENSITY_COLORS = {
    "high-intensity":   "#e74c3c",
    "medium-intensity": "#f39c12",
    "low-intensity":    "#3498db",
    "unknown":          "#95a5a6",
}

if NNSIGHT_AVAILABLE:
    _tok_preview = AutoTokenizer.from_pretrained(MODEL_NAME)
    _preview_ids = _tok_preview.encode(INPUT_TEXT)
    _preview_strs = [_tok_preview.decode([t]) for t in _preview_ids]
    print(f"Input:   {INPUT_TEXT!r}")
    print(f"Tokens:  {_preview_strs}")
    print(f"SEQ_LEN: {len(_preview_ids)}")
    print(f"Total decode calls: {len(_preview_ids)} tokens x {N_LAYERS} layers = {len(_preview_ids) * N_LAYERS}")
    del _tok_preview, _preview_ids, _preview_strs
else:
    print("Skipping preview - nnsight not available")

## API & extractor initialization

`HypoSpaceAPI` is the single integration point for decoding and governance.
`NNSightExtractor` wraps an `nnsight.LanguageModel` and provides a disk-cached
multi-layer single-pass `extract_layers()` API.

In [ ]:
api      = HypoSpaceAPI(config=DecoderConfig(
    top_k=TOP_K,
    runtime=RuntimeConfig(device="cpu", cache_dir=CACHE_DIR),
))
canvas   = SemanticCanvas()
analyzer = MechanisticAnalyzer()
print("HypoSpaceAPI + MechanisticAnalyzer ready")

if NNSIGHT_AVAILABLE:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    extractor = NNSightExtractor(MODEL_NAME, device="cpu", cache_dir=CACHE_DIR)

    token_ids  = tokenizer.encode(INPUT_TEXT)
    TOKEN_STRS = [tokenizer.decode([t]) for t in token_ids]
    SEQ_LEN    = len(TOKEN_STRS)

    print(f"Model:   {MODEL_NAME}")
    print(f"Input:   {INPUT_TEXT!r}")
    print(f"Tokens:  {list(enumerate(TOKEN_STRS))}")
    print(f"SEQ_LEN: {SEQ_LEN}")
    print(f"Layers:  {N_LAYERS}  ({LAYER_PATHS[0]} ... {LAYER_PATHS[-1]})")
    print(f"Total (token x layer) pairs: {SEQ_LEN * N_LAYERS}")

## Section 1 - Multi-layer, multi-token extraction

`NNSightExtractor.extract_layers(inputs, layer_paths, token_index)` runs ONE forward pass
and captures all 12 layer activations at one token position. To cover every token position
we loop over `token_index`; this gives `SEQ_LEN` forward passes total, each yielding 12 vectors.

Activations are cached under `.hypo_cache/nnsight/` with key
`(model_name, layer_path, inputs, token_index)`. Re-running this cell is instantaneous.

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    print(f"Extracting: {SEQ_LEN} forward passes, {N_LAYERS} layers each")
    print("(activations cached on disk after first run)\n")

    raw_acts = {}
    for tok_idx in range(SEQ_LEN):
        raw_acts[tok_idx] = extractor.extract_layers(
            INPUT_TEXT, LAYER_PATHS, token_index=tok_idx,
        )
        vecs = raw_acts[tok_idx]
        dims = len(next(iter(vecs.values())))
        print(f"  token {tok_idx} {TOKEN_STRS[tok_idx]!r:14s}: {len(vecs)} layers x {dims} dims")

    print(f"\nTotal activation vectors: {SEQ_LEN * N_LAYERS}")

## Section 2 - Decode + mechanistic intervention

For each (token, layer) activation vector we call `api.decode()` to extract top-k features,
then run `MechanisticAnalyzer.run_interventions()` to get effect sizes for those features.
The two steps are merged into one loop to avoid duplicating decode calls.

Each `Feature` carries:
- `source_index`: the neuron dimension index in the 768-dim hidden state - the **cluster key**
- `score`: max-abs normalized activation magnitude
- `label`: human-readable intensity-band description from `SemanticInterpreter`

Note: `MechanisticAnalyzer` here is the CPU-only stub (`effect_size = score * 0.5`).
For real zero-ablation interventions, swap to `decode_and_score_from_model()` which
uses `PyVeneInterventionRunner` - but that requires one ablated forward pass per feature
(SEQ_LEN * N_LAYERS * TOP_K passes total), which is too slow for an interactive notebook.

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="No prior kernel found.*")

        records = []
        for tok_idx in range(SEQ_LEN):
            for layer_idx, (lp, lid) in enumerate(zip(LAYER_PATHS, LAYER_IDS)):
                decode_result = api.decode(
                    MODEL_NAME, lid, raw_acts[tok_idx][lp], version=VERSION,
                )
                interventions = analyzer.run_interventions(decode_result.features)
                effect_by_id  = {iv.feature_id: iv.effect_size for iv in interventions}

                for feat in decode_result.features:
                    intensity = feat.label.split()[0] if feat.label else "unknown"
                    records.append({
                        "token_idx":    tok_idx,
                        "token_str":    TOKEN_STRS[tok_idx],
                        "layer_idx":    layer_idx,
                        "layer_path":   lp,
                        "feature_id":   feat.id,
                        "source_index": feat.source_index,
                        "score":        feat.score,
                        "label":        feat.label or "unknown",
                        "intensity":    intensity,
                        "effect_size":  effect_by_id.get(feat.id, 0.0),
                    })

    df = pd.DataFrame(records)
    print(f"Records:                  {len(df)}")
    print(f"Unique neuron dims fired: {df['source_index'].nunique()}")
    print(f"\nIntensity distribution:")
    print(df["intensity"].value_counts().to_string())

## Section 3 - 3D semantic & mechanistic concept map

The main visualization. Every dot is one (feature, token, layer) record:
- **X** = token position
- **Y** = layer depth (h.0 = bottom, h.11 = top)
- **Z** = feature score
- **Color** = intensity band (legend toggles isolate bands)
- **Size** = mechanistic effect size

Hover any point to see its `feature_id`, neuron dimension, label, and effect size.
Use the legend on the right to isolate a single intensity band.

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    traces = []
    for band, color in INTENSITY_COLORS.items():
        sub = df[df["intensity"] == band]
        if sub.empty:
            continue
        hover = (
            "<b>" + sub["feature_id"] + "</b><br>"
            + "label: " + sub["label"] + "<br>"
            + "neuron dim: " + sub["source_index"].astype(str) + "<br>"
            + "layer: " + sub["layer_path"] + "<br>"
            + "token: " + sub["token_str"].astype(str) + " (pos " + sub["token_idx"].astype(str) + ")<br>"
            + "score: " + sub["score"].round(4).astype(str) + "<br>"
            + "effect_size: " + sub["effect_size"].round(4).astype(str)
        )
        traces.append(go.Scatter3d(
            x=sub["token_idx"],
            y=sub["layer_idx"],
            z=sub["score"],
            mode="markers",
            name=band,
            marker=dict(
                size=4 + sub["effect_size"] * 20,
                color=color,
                opacity=0.75,
                line=dict(width=0.5, color="white"),
            ),
            text=hover,
            hovertemplate="%{text}<extra></extra>",
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f'GPT-2 3D Semantic Concept Map - "{INPUT_TEXT}"',
        scene=dict(
            xaxis=dict(
                title="Token Position",
                tickvals=list(range(SEQ_LEN)),
                ticktext=TOKEN_STRS,
            ),
            yaxis=dict(
                title="Layer Depth",
                tickvals=list(range(N_LAYERS)),
                ticktext=LAYER_LABELS,
            ),
            zaxis=dict(title="Feature Score"),
        ),
        legend_title="Intensity Band",
        height=700,
    )
    fig.show()

## Section 4 - Neuron cluster identification

A **neuron cluster** is the set of all feature records sharing the same `source_index`.
This represents one neuron in the 768-dim hidden state, observed across all (token, layer)
positions where it ranked in the top-k.

For each cluster we compute:
- `n_positions`: how many (token, layer) positions it fired in
- `total_score`: sum of all its scores - the ranking metric
- `score_trajectory`: max score per token position (across layers), padded with 0.0
- `layer_spread`: how many distinct layers it appears in
- `dominant_intensity`: the most common intensity band for that cluster

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    cluster_stats = []
    for src_idx, group in df.groupby("source_index"):
        traj = (
            group.groupby("token_idx")["score"]
            .max()
            .reindex(range(SEQ_LEN), fill_value=0.0)
            .tolist()
        )
        cluster_stats.append({
            "source_index":       int(src_idx),
            "n_positions":        len(group),
            "mean_score":         round(float(group["score"].mean()), 4),
            "max_score":          round(float(group["score"].max()), 4),
            "total_score":        round(float(group["score"].sum()), 4),
            "mean_effect_size":   round(float(group["effect_size"].mean()), 4),
            "layer_spread":       int(group["layer_idx"].nunique()),
            "layers_active":      sorted(int(x) for x in group["layer_idx"].unique()),
            "score_trajectory":   traj,
            "dominant_intensity": group["intensity"].mode().iloc[0],
        })

    clusters_df  = pd.DataFrame(cluster_stats).sort_values("total_score", ascending=False).reset_index(drop=True)
    top_clusters = clusters_df.head(TOP_N_CLUSTERS).reset_index(drop=True)

    print(f"Total unique neuron clusters: {len(clusters_df)}")
    print(f"\nTop {TOP_N_CLUSTERS} by total activation:")
    cols = ["source_index", "n_positions", "mean_score", "max_score", "layer_spread", "dominant_intensity"]
    print(top_clusters[cols].to_string(index=False))

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    row_bg = [INTENSITY_COLORS.get(v, "#ecf0f1") for v in top_clusters["dominant_intensity"]]

    fig = go.Figure(go.Table(
        header=dict(
            values=["Neuron Dim", "# Positions", "Mean Score", "Max Score", "Layer Spread", "Dominant Intensity"],
            fill_color="steelblue",
            font=dict(color="white", size=13),
            align="center",
        ),
        cells=dict(
            values=[
                top_clusters["source_index"],
                top_clusters["n_positions"],
                top_clusters["mean_score"].round(4),
                top_clusters["max_score"].round(4),
                top_clusters["layer_spread"],
                top_clusters["dominant_intensity"],
            ],
            fill_color=[row_bg] * 6,
            align="center",
            font=dict(size=12),
        ),
    ))
    fig.update_layout(
        title=f"Top {TOP_N_CLUSTERS} Neuron Clusters - ranked by total activation",
        height=420,
    )
    fig.show()

## Section 5 - Temporal evolution: cluster score trajectories

For each top-N neuron cluster, plot its max-score-across-layers as a function of
token position. This reveals which neurons fire at which points during inference.

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    fig = go.Figure()
    for _, crow in top_clusters.iterrows():
        src   = int(crow["source_index"])
        traj  = crow["score_trajectory"]
        color = INTENSITY_COLORS.get(crow["dominant_intensity"], "#95a5a6")
        fig.add_trace(go.Scatter(
            x=list(range(SEQ_LEN)),
            y=traj,
            mode="lines+markers",
            name=f"dim {src}",
            line=dict(color=color, width=2),
            marker=dict(size=7),
            hovertemplate=(
                f"<b>Neuron dim {src}</b><br>"
                "token pos: %{x}<br>"
                "max score: %{y:.4f}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=(
            f"Neuron Cluster Score Trajectories - top {TOP_N_CLUSTERS}<br>"
            f'<sup>"{INPUT_TEXT}"</sup>'
        ),
        xaxis=dict(
            title="Token Position",
            tickvals=list(range(SEQ_LEN)),
            ticktext=TOKEN_STRS,
        ),
        yaxis=dict(title="Max Score Across Layers"),
        legend_title="Neuron Dimension",
        height=520,
    )
    fig.show()

## Section 6 - 3D cluster trajectories

Each top cluster's path through (token, layer, score) space, drawn as a connected line.
This exposes "deep-firing" neurons (active across many layers at one token) vs
"broad-firing" neurons (active across many tokens at similar layer depths).

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    fig = go.Figure()
    for _, crow in top_clusters.iterrows():
        src = int(crow["source_index"])
        sub = df[df["source_index"] == src].sort_values(["token_idx", "layer_idx"])
        fig.add_trace(go.Scatter3d(
            x=sub["token_idx"],
            y=sub["layer_idx"],
            z=sub["score"],
            mode="lines+markers",
            name=f"dim {src}",
            marker=dict(size=5, opacity=0.9),
            line=dict(width=3),
            hovertemplate=(
                f"<b>Neuron dim {src}</b><br>"
                "token: %{x}<br>"
                "layer: %{y}<br>"
                "score: %{z:.4f}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=f"3D Neuron Cluster Trajectories - top {TOP_N_CLUSTERS}",
        scene=dict(
            xaxis=dict(title="Token Position", tickvals=list(range(SEQ_LEN)), ticktext=TOKEN_STRS),
            yaxis=dict(title="Layer Depth", tickvals=list(range(N_LAYERS)), ticktext=LAYER_LABELS),
            zaxis=dict(title="Feature Score"),
        ),
        height=650,
    )
    fig.show()

## Section 7 - Interactive cluster isolation

Pick any of the top-N clusters from the dropdown to see:
- A `(layer x token)` heatmap of that neuron's score wherever it ranked in top-k
- A detail table of every record where this neuron appeared

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    import ipywidgets as widgets
    from IPython.display import display

    cluster_options = [
        (
            f"dim {int(row['source_index'])} - {row['dominant_intensity']} ({int(row['n_positions'])} positions)",
            int(row["source_index"]),
        )
        for _, row in top_clusters.iterrows()
    ]

    dropdown = widgets.Dropdown(
        options=cluster_options,
        description="Cluster:",
        layout=widgets.Layout(width="70%"),
    )
    output = widgets.Output()

    def _render(src):
        with output:
            output.clear_output(wait=True)
            sub = df[df["source_index"] == src]

            pivot = sub.pivot_table(
                index="layer_idx", columns="token_idx",
                values="score", aggfunc="max", fill_value=0.0,
            )
            pivot = pivot.reindex(range(N_LAYERS), fill_value=0.0)
            pivot = pivot.reindex(columns=range(SEQ_LEN), fill_value=0.0)

            fig = go.Figure(go.Heatmap(
                z=pivot.values,
                x=[TOKEN_STRS[i] for i in range(SEQ_LEN)],
                y=LAYER_LABELS,
                colorscale="Plasma",
                text=pivot.values.round(4),
                texttemplate="%{text}",
                hovertemplate="layer: %{y}<br>token: %{x}<br>score: %{z:.4f}<extra></extra>",
                colorbar=dict(title="Score"),
            ))
            fig.update_layout(
                title=f"Neuron dim {src} - score heatmap (layer x token)",
                xaxis_title="Token",
                yaxis_title="Layer",
                height=420,
            )
            fig.show()

            detail = (
                sub[["token_str", "layer_path", "feature_id", "score", "effect_size", "label"]]
                .sort_values("score", ascending=False)
                .reset_index(drop=True)
            )
            print(detail.to_string(index=False))

    def on_select(change):
        _render(change["new"])

    dropdown.observe(on_select, names="value")
    display(dropdown, output)
    _render(int(top_clusters.iloc[0]["source_index"]))

## Section 8 - Layer spread per token

For each token position, count how many distinct layers each top-N cluster fires in.
Tall bars = neurons that fire deeply through the network at that token.

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    plot_data = []
    top_dim_set = set(top_clusters["source_index"].astype(int).tolist())
    for tok_idx in range(SEQ_LEN):
        tok_sub = df[(df["token_idx"] == tok_idx) & df["source_index"].isin(top_dim_set)]
        for src, grp in tok_sub.groupby("source_index"):
            plot_data.append({
                "token_str":    TOKEN_STRS[tok_idx],
                "source_index": str(int(src)),
                "layer_count":  int(grp["layer_idx"].nunique()),
            })

    cdf = pd.DataFrame(plot_data)
    if cdf.empty:
        print("No top-cluster firings to plot.")
    else:
        fig = px.bar(
            cdf, x="token_str", y="layer_count",
            color="source_index",
            barmode="group",
            title=f"Layer Spread per Token - top {TOP_N_CLUSTERS} neuron clusters",
            labels={
                "token_str":    "Token",
                "layer_count":  "# Layers Active",
                "source_index": "Neuron Dim",
            },
            height=440,
        )
        fig.show()

## Section 9 - Per-layer feature score distributions

How does the score distribution shift from early to late layers?
One histogram per transformer block, all on a shared score axis.

In [ ]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    fig = make_subplots(
        rows=2, cols=6,
        subplot_titles=LAYER_LABELS,
        shared_xaxes=True,
        shared_yaxes=True,
    )
    for layer_idx in range(N_LAYERS):
        row = layer_idx // 6 + 1
        col = layer_idx % 6 + 1
        sub = df[df["layer_idx"] == layer_idx]
        fig.add_trace(
            go.Histogram(x=sub["score"], nbinsx=20, marker_color="#3498db",
                         opacity=0.75, showlegend=False),
            row=row, col=col,
        )
    fig.update_layout(
        title_text="Feature Score Distributions Across All 12 Layers",
        height=500,
    )
    fig.show()

## Section 10 - Diagnostics & summary

In [ ]:
report = run_diagnostics()
print(f"Diagnostics - overall: {report.overall_status}")
for probe in report.probes:
    icon = {"ok": "OK  ", "degraded": "WARN", "error": "ERR "}.get(probe.status, "?   ")
    print(f"  [{icon}] {probe.subsystem:32s} ({probe.latency_ms:.1f} ms)")

print()
if NNSIGHT_AVAILABLE:
    print("Notebook summary:")
    print(f"  Input text:          {INPUT_TEXT!r}")
    print(f"  Tokens:              {SEQ_LEN}")
    print(f"  Layers:              {N_LAYERS}")
    print(f"  Feature records:     {len(df)}")
    print(f"  Unique neuron dims:  {df['source_index'].nunique()}")
    print(f"  Neuron clusters:     {len(clusters_df)}")
    print(f"  Top clusters shown:  {TOP_N_CLUSTERS}")
    print(f"  Forward passes:      {SEQ_LEN} (one per token position)")
else:
    print("nnsight not available - install torch + nnsight to run GPT-2 sections")